In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 XGBoost Classifier (`models/xgboost_raw_esi1_extreme.ipynb`)

This notebook trains a **Binary XGBoost Classifier** for **ESI 1 vs Not ESI 1** using **17 Features** (7 raw triage inputs + 10 binary vital anomaly flags; strictly excluding non-binary feature engineered deltas/ranges and vital history min/max/last columns):

### System Architecture & Workflow
1. **Stratified Partitioning First**: Splits the dataset into Train (70%), Validation (15%), and Test (15%) splits before resampling to prevent data leakage.
2. **Predictor Feature Selection (17 Total Features)**:
   - **7 Raw Triage Inputs**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`.
   - **10 Binary Vital Anomaly Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
3. **Factor-Controlled Class Downsampling**: Regulates majority class representation on `train_df` (`undersample_ratio = 1.0`).
4. **Binary XGBoost Gradient Boosting**: Fits decision trees with `objective = "binary:logistic"` and `eval_metric = "logloss"`.
5. **Full Metrics Suite Evaluation**: Computes Accuracy, **Balanced Accuracy**, **Specificity**, Precision, Recall/Sensitivity, F1 Score, ROC-AUC, and **MCC Score**.
6. **Reports & Artifacts**:
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_raw_esi1_metrics_barchart.png`).
   - **CSV Reports**: `reports/xgboost_raw_esi1_val_report.csv`, `reports/xgboost_raw_esi1_test_report.csv`.
   - **Model Export**: Saved to `deploy/xgboost_raw_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 17 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
t_hr  <- get_vec("triage_vital_hr")
t_sbp <- get_vec("triage_vital_sbp")
t_o2  <- get_vec("triage_vital_o2")
t_rr  <- get_vec("triage_vital_rr")
# Construct 17 Predictor Features (7 raw triage + 10 binary vital anomaly flags)
df_full <- data.frame(
  # 7 Raw Triage Inputs
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  
  # 10 Binary Vital Anomaly Flags
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Complete Case Dataset Ready (17 Features): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Natural Binary Target Distribution ('1' vs 'not_1'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning FIRST & Class Downsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)
undersample_ratio <- 1.0
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
cont_cols <- c("age", "triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2")
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
# Apply Downsampling to Training Set Only
idx_1     <- which(train_df$target_layer1 == "1")
idx_not_1 <- which(train_df$target_layer1 == "not_1")
n_esi1       <- length(idx_1)
n_not1_keep  <- min(as.integer(n_esi1 * undersample_ratio), length(idx_not_1))
kept_not1_idx <- sample(idx_not_1, size = n_not1_keep)
train_df     <- train_df[sort(c(idx_1, kept_not1_idx)), ]
cat(sprintf("Downsampled Training Target Distribution ('1' vs 'not_1'):\n"))
print(table(train_df$target_layer1))
feat_names <- setdiff(names(train_df), "target_layer1")
X_train <- as.matrix(train_df[, feat_names])
y_train <- ifelse(train_df$target_layer1 == "1", 1, 0)
X_val   <- as.matrix(val_df[, feat_names])
y_val   <- ifelse(val_df$target_layer1 == "1", 1, 0)
X_test  <- as.matrix(test_df[, feat_names])
y_test  <- ifelse(test_df$target_layer1 == "1", 1, 0)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary XGBoost Model
# ---------------------------------------------------------
set.seed(config$training$random_state)
dtrain <- xgb.DMatrix(data = X_train, label = y_train)
dval   <- xgb.DMatrix(data = X_val,   label = y_val)
dtest  <- xgb.DMatrix(data = X_test,  label = y_test)
xgb_params <- list(
  objective        = "binary:logistic",
  eval_metric      = "logloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
model_esi1 <- xgb.train(
  params                = xgb_params,
  data                  = dtrain,
  nrounds               = 200,
  watchlist             = list(train = dtrain, val = dval),
  early_stopping_rounds = 25,
  verbose               = 0
)
cat("XGBoost ESI 1 Detector Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmarking on Validation & Test Sets
# ---------------------------------------------------------
eval_split <- function(X_mat, y_true, split_name) {
  dmat <- xgb.DMatrix(data = X_mat)
  raw_probs <- predict(model_esi1, dmat)
  pred_fac  <- factor(ifelse(raw_probs >= 0.5, "1", "not_1"), levels = c("1", "not_1"))
  act_fac   <- factor(ifelse(y_true == 1, "1", "not_1"), levels = c("1", "not_1"))
  
  cm   <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc  <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  spec <- as.numeric(cm$byClass["Specificity"])
  bal_acc <- as.numeric(cm$byClass["Balanced Accuracy"])
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  r_obj <- tryCatch(pROC::roc(y_true, raw_probs), error = function(e) NULL)
  auc_score <- if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  
  tp <- sum(pred_fac == "1" & act_fac == "1")
  tn <- sum(pred_fac == "not_1" & act_fac == "not_1")
  fp <- sum(pred_fac == "1" & act_fac == "not_1")
  fn <- sum(pred_fac == "not_1" & act_fac == "1")
  
  num   <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  mcc   <- if (is.na(denom) || denom == 0) 0 else num / denom
  
  cat(sprintf("=== %s Benchmark ===\n", split_name))
  cat(sprintf("  Accuracy          : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Balanced Accuracy : %.4f\n", bal_acc))
  cat(sprintf("  Specificity       : %.4f\n", spec))
  cat(sprintf("  Precision         : %.4f\n", prec))
  cat(sprintf("  Recall (Sens)     : %.4f\n", rec))
  cat(sprintf("  F1-Score          : %.4f\n", f1))
  cat(sprintf("  ROC-AUC           : %.4f\n", auc_score))
  cat(sprintf("  MCC Score         : %.4f\n\n", mcc))
  
  return(data.frame(
    Split = split_name, Accuracy = round(acc, 4), Balanced_Accuracy = round(bal_acc, 4),
    Specificity = round(spec, 4), Precision = round(prec, 4), Recall_Sensitivity = round(rec, 4),
    F1_Score = round(f1, 4), ROC_AUC = round(auc_score, 4), MCC_Score = round(mcc, 4)
  ))
}
val_rep  <- eval_split(X_val, y_val, "Validation")
test_rep <- eval_split(X_test, y_test, "Test")
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(val_rep,  file = file.path(reports_dir, "xgboost_raw_esi1_val_report.csv"), row.names = FALSE)
write.csv(test_rep, file = file.path(reports_dir, "xgboost_raw_esi1_test_report.csv"), row.names = FALSE)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots & Save Model Artifact
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = model_esi1, preproc = preproc), file = file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds"))
cat("Layer 1 XGBoost ESI 1 Detector saved to deploy/xgboost_raw_esi1_extreme_model.rds\n")